# Module 1.1: Hardware Evaluation, Safe Battery Removal & 24/7 AC Power Bypass

## Homelab Expansion (Chapter 2) - Step 1: Hardware Refurbishment & Battery Bypass

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Understand the severe fire and physical damage hazards ("spicy pillow" phenomenon) of leaving a degraded Lithium-Polymer (Li-Po) battery plugged in 24/7.
2. Assemble the complete hardware repair, battery bypass, and thermal maintenance shopping list for a 2019 ASUS ROG Zephyrus (`GA502` / `GU502` / `GX502`).
3. Safely open the Zephyrus chassis, disconnect the motherboard battery connector without shorting nearby SMD capacitors, and extract the degraded battery pack.
4. Insulate the exposed battery header and overcome the ASUS Embedded Controller (EC) "dead boot" quirk when booting strictly on AC barrel power.
5. Mitigate missing battery transient power bridging using an external Uninterruptible Power Supply (UPS) and Linux GPU power limits.

## 1.1.1 Conceptual Background: Why Remove & Bypass a Degraded Laptop Battery?

When repurposing a 2019 gaming laptop like the **ASUS ROG Zephyrus** (`GA502`, `GU502`, or `GX502`, equipped with an `C41N1837` / `76Wh` 4-cell Li-Po pouch battery) into an always-on 24/7 server, leaving a worn-out battery inside the chassis is the single biggest reliability and safety hazard in a homelab.

### 1. The Lithium-Polymer "Spicy Pillow" Hazard
Unlike cylindrical `18650` cells, thin gaming laptops use soft aluminum-laminated pouch cells. When a degraded pouch battery is held at 100% trickle charge inside a warm chassis running 24/7 workloads, electrolyte decomposition generates flammable gases ($	ext{CO}_2$, $	ext{CO}$, and hydrocarbons). This causes the pouch to swell:
- **Mechanical Damage:** On the 2019 ROG Zephyrus, the battery sits directly beneath the trackpad and palmrest. A swelling battery bends the magnesium-alloy deck, cracks the trackpad glass, and stresses the motherboard PCB traces.
- **Thermal Runaway:** A punctured or severely swollen Li-Po pouch can enter thermal runaway. Removing the battery entirely eliminates this hazard permanently.

### 2. How the Laptop "Battery Bypass" Actually Works
Modern laptops use a buck-boost DC-DC charging IC (Narrow VDC / Hybrid Power Boost architecture) controlled by the motherboard's **Embedded Controller (EC)**:

```mermaid
flowchart LR
    UPS["External Pure Sine-Wave UPS<br>(Replaces Internal Battery)"] ==>|"120V AC"| Brick["OEM ASUS AC Power Brick<br>(180W / 230W DC Barrel)"]
    Brick ==>|"19.5V - 20V DC"| ChargerIC["Motherboard Charger IC<br>& Power Rail MOSFETs"]
    
    subgraph Laptop ["2019 ROG Zephyrus Motherboard"]
        ChargerIC ==>|"Direct Primary System Rail (VSYS)"| VRM["CPU & NVIDIA GPU VRMs"]
        BattHeader["Internal Battery Header<br>(Unplugged & Kapton-Insulated)"] -. "Open Circuit (Bypassed)" .- ChargerIC
        EC["Embedded Controller (EC)<br>(Monitors Adapter Wattage)"] --> VRM
    end
```

- **No Solder Jumpers Required:** You do **not** solder wires across the battery terminals! Soldering across a lithium battery connector will dead-short the charging MOSFETs and destroy the motherboard. The Zephyrus motherboard natively routes 19.5V/20V DC from the barrel jack straight to the `VSYS` system power rail when the battery connector is left open (unplugged).
- **The Two Engineering Caveats of Running Battery-Less:**
  1. **The ASUS EC "Dead Boot" Quirk:** The first time AC power is plugged in after removing the battery, the Embedded Controller may refuse to power on because its CMOS/EC state registers a missing battery fault. Performing a **40-second EC Hard Reset** clears this latch.
  2. **Transient Power Bridging (Hybrid Power Boost):** Under extreme gaming spikes, ASUS laptops briefly pull 10W–20W from the battery to assist the AC brick. Without a battery, sudden 100% CPU + 100% GPU spikes can trip the AC adapter's Over-Current Protection (OCP) or trigger PROCHOT throttling. We solve this cleanly in software (capping the NVIDIA GPU power limit by 15%–20% via `nvidia-smi -pl` in Module 2.2) and externally via a **UPS**.

## 🛒 Hardware Repair & Battery Bypass Shopping List

Before opening the laptop, make sure you have the following tools and consumables on your workbench:

| Category | Item & Specification | Why You Need It |
| :--- | :--- | :--- |
| **Driver Kit** | **Precision Phillips `PH0` & `PH1` Screwdriver + Plastic Spudger** | Removes the 14–15 bottom cover screws and 4–6 internal battery frame screws without stripping heads or scratching the chassis. |
| **Insulation** | **10mm Kapton (Polyimide) Electrical Tape** | Covers the exposed motherboard battery connector pins and secures any loose sensor cables after battery extraction. |
| **ESD Safety** | **Nylon / ESD-Safe Angled Tweezers** | Safely slides back the metal locking bail on the ASUS battery connector without risking a metal screwdriver slip across live SMD components. |
| **Thermal Refresh** | **Arctic MX-6 / Noctua NT-H2 (or PTM7950) + `K5 PRO` Viscous Thermal Putty + 99% Isopropyl Alcohol** | While the bottom cover is off (Module 1.2), you will clean 7 years of dust and replace dried 2019 factory paste and VRM thermal putty. |
| **External Bypass** | **600VA–1000VA Pure Sine-Wave UPS** *(e.g., CyberPower CP1000PFCLCD or APC Back-UPS)* | Acts as your external, fire-safe battery outside the laptop chassis—preventing sudden power-loss crashes and dirty AC brownouts. |

## 1.1.2 Step-by-Step Laboratory Guide

### Task 1.1.1: Pre-Teardown Safety & Screw Mapping
1. Shut down the ROG Zephyrus completely.
2. Unplug the OEM AC power barrel adapter and remove all USB peripherals.
3. Press and hold the **Power Button for 15 seconds** to discharge residual capacitance in the motherboard rails.
4. Flip the laptop upside down onto an anti-static mat or clean microfiber towel.
5. **Crucial Zephyrus Screw Map:** On 2019 Zephyrus models (`GA502` / `GU502` / `GX502`), the bottom panel uses **different screw lengths** (longer screws along the rear hinge/exhaust vents, shorter screws along the front lip, and often one captive pop-up screw on the bottom-right corner). Draw a quick grid on a sheet of paper and place each screw on its matching spot as you remove it.
6. Use your plastic spudger starting at the captive corner gap to gently pop the perimeter clips and lift away the bottom panel.

### 🖼️ Task 1.1.1 Verification: Initial Internal Hardware Inspection

**Verification Objective:** Capture a top-down photo of the opened ROG Zephyrus interior showing the original battery pack, dual cooling fans, and copper heatpipes before modification.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification photo below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

### Task 1.1.2: Safe Battery Connector Release & Pack Extraction

> ⚠️ **CRITICAL MOTHERBOARD WARNING:** The 2019 ASUS ROG Zephyrus uses a vertical/slide-lock 8-pin battery connector surrounded by tiny surface-mount capacitors. **Never use a metal flathead screwdriver to pry the battery plug!** A 1mm slip with metal while the battery is still plugged in will short the 15.4V battery rail directly to ground or the EC data lines, permanently killing the motherboard.

1. **Slide Back the Metal Locking Clip:** Inspect where the multi-wire battery cable meets the motherboard. Using a fingernail or plastic spudger, gently slide the small silver metal retaining tab **horizontally away from the battery plug** (toward the motherboard/heatsink) by ~2mm until it unlatches.
2. **Lift Vertically:** Grab the black plastic connector housing (or pull evenly on both sides with plastic fingers/spudgers) and lift **straight up** vertically off the motherboard pins.
3. **Unscrew the Battery Frame:** Remove the Phillips screws securing the black `C41N1837` Li-Po battery frame to the palmrest chassis.
4. **Lift & Inspect:** Carefully lift the battery out of the laptop. (If the cells are swollen, do not bend or compress the pouch). Place it in a non-conductive container to take to a certified e-waste/battery drop-off center (e.g., Best Buy, Home Depot Call2Recycle, or municipal household hazardous waste).
5. **Insulate the Header:** Apply a small strip of **Kapton tape** over the exposed motherboard battery connector housing so no stray metallic debris can ever bridge the pins.

### 🖼️ Task 1.1.2 Verification: Empty Battery Bay & Insulated Header

**Verification Objective:** Capture a photo of the laptop interior with the Li-Po battery completely removed and the motherboard battery connector cleanly unlatched/insulated.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification photo below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

### Task 1.1.3: Embedded Controller (EC) Reset & Battery-Less AC Boot Test
Before proceeding to thermal maintenance in Module 1.2, verify that the laptop boots cleanly on pure AC wall power:
1. Temporarily set the bottom cover back in place (no need to tighten all screws yet, as we will service the heatsinks in Module 1.2) and flip the laptop right-side up.
2. **Perform the ASUS EC Reset (with AC Unplugged):** Hold down the **Power Button continuously for 45 seconds**. This resets the Embedded Controller's battery-detection registers.
3. **Connect the OEM Barrel Charger (via your UPS):** Plug the original high-wattage ASUS barrel power brick (180W/230W—do *not* use a 65W USB-C PD charger as your primary power source without a battery!) into the laptop.
4. Press the **Power Button** and immediately tap `F2` (or `DEL`) repeatedly to enter the **ASUS ROG UEFI BIOS Utility**.
5. Confirm that the system posts cleanly on AC power and shows the battery icon as disconnected/absent.

### 🖼️ Task 1.1.3 Verification: Battery-Less AC Boot into UEFI BIOS

**Verification Objective:** Capture a photo of the ROG Zephyrus screen booted into the ASUS UEFI BIOS Utility running exclusively on the AC power adapter with the internal battery removed.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification photo below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

## 🏁 Module Checkpoint & Reflection

Before proceeding to the next guide, confirm the following checklist:

- [ ] Bottom panel screws mapped by position and length.
- [ ] Degraded Li-Po battery safely unlatched, unscrewed, removed, and staged for certified e-waste recycling.
- [ ] Motherboard battery header inspected and covered with Kapton tape.
- [ ] EC reset completed and battery-less AC boot into ASUS UEFI BIOS verified.
- [ ] Verification photos captured and attached above.

Proceed to [Module 1.2: Deep Thermal Maintenance & BIOS Server Tuning](./01_2_thermal_maintenance_and_bios_prep.ipynb).